In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ======================
# Load Data
# ======================
train = pd.read_csv("/kaggle/input/titanic/train.csv")
test = pd.read_csv("/kaggle/input/titanic/test.csv")

test_ids = test["PassengerId"]

full = pd.concat([train, test], sort=False)

# ======================
# Feature Engineering
# ======================

# Fill basic missing values
full["Fare"] = full["Fare"].fillna(full["Fare"].median())
full["Embarked"] = full["Embarked"].fillna(full["Embarked"].mode()[0])
full["Cabin"] = full["Cabin"].fillna("U")

# Extract Title
full["Title"] = full["Name"].str.extract(" ([A-Za-z]+).", expand=False)

# Replace rare titles
rare_titles = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
full["Title"] = full["Title"].replace(rare_titles, "Rare")
full["Title"] = full["Title"].replace({"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"})

# Fill missing titles safely
full["Title"] = full["Title"].fillna("Rare")

# Family features
full["FamilySize"] = full["SibSp"] + full["Parch"] + 1
full["IsAlone"] = (full["FamilySize"] == 1).astype(int)

# Fill Age by Title median
full["Age"] = full.groupby("Title")["Age"].transform(lambda x: x.fillna(x.median()))

# If still any Age missing (safety)
full["Age"] = full["Age"].fillna(full["Age"].median())

# Drop unused columns
full.drop(["PassengerId","Name","Ticket","Cabin"], axis=1, inplace=True)

# Convert categorical to numeric
full = pd.get_dummies(full)

# FINAL SAFETY: remove any remaining NaN
full = full.fillna(0)

# ======================
# Split Back
# ======================
train_processed = full.iloc[:len(train)]
test_processed = full.iloc[len(train):]

X_train = train_processed.drop("Survived", axis=1)
y_train = train_processed["Survived"].astype(int)
X_test = test_processed.drop("Survived", axis=1)

# ======================
# Optimized SVM
# ======================
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(
        C=10,
        gamma=0.01,
        kernel="rbf",
        probability=True,
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)
pred = svm_model.predict(X_test)

# ======================
# Submission
# ======================
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": pred.astype(int)
})

submission.to_csv("submission.csv", index=False)